In [2]:
import sys
sys.path.insert(0, '..')

from src.indexing.embeddings import embed_query
from src.indexing.vector_store import load_vector_store, query_collection
from src.generation.answer_generator import generate_answer, build_prompt
from src.generation.citations import (
    format_citations, validate_citations, format_answer_with_citations
)

# Load the ChromaDB collection indexed on Day 5
collection = load_vector_store(
    persist_dir='../vector_store',
    collection_name='sr_papers',
)
print(f'Collection loaded: {collection.count()} chunks')

Collection loaded: 695 chunks


In [3]:
def rag_query(question, top_k=5, provider='mock'):
    """Retrieve + generate + display in one call."""
    # Step 1: retrieve
    qv = embed_query(question)
    chunks = query_collection(collection, qv, top_k=top_k)

    # Step 2: generate
    result = generate_answer(question, chunks, provider=provider)

    # Step 3: validate citations
    report = validate_citations(result['answer'], result['sources'])

    # Step 4: display
    print('=' * 65)
    print(f'Q: {question}')
    print('=' * 65)
    print(format_answer_with_citations(result['answer'], result['sources']))
    print()
    print(f"Provider  : {result['provider']} / {result['model']}")
    print(f"Latency   : {result['latency_ms']:.0f} ms")
    tokens = result['token_usage']
    print(f"Tokens    : {tokens['total_tokens']} (prompt={tokens['prompt_tokens']}, completion={tokens['completion_tokens']})")
    print(f"Est. cost : ${tokens['estimated_cost_usd']:.6f}")
    print(f"Citations : {'VALID' if report['valid'] else 'ISSUES FOUND'}")
    if not report['valid']:
        print(f"  Missing indices: {report['missing']}")
    if report['unused_sources']:
        print(f"  Unused sources: {report['unused_sources']}")
    print()
    return result

result = rag_query('What loss function does SRGAN use?')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11248.30it/s]


Q: What loss function does SRGAN use?
Based on the provided context, the super-resolution methods use various loss functions and architectural components. [1] describes the use of perceptual loss combined with adversarial training. [2] introduces residual channel attention to focus on informative features. This is a mock response — set LLM_PROVIDER in .env to get real answers.

**References**
[1] SRGAN / SRResNet (, 2016) — srgan_2016.pdf, page 7
[2] SRGAN / SRResNet (, 2016) — srgan_2016.pdf, page 7
[3] SRGAN / SRResNet (, 2016) — srgan_2016.pdf, page 7
[4] SRGAN / SRResNet (, 2016) — srgan_2016.pdf, page 6
[5] SRGAN / SRResNet (, 2016) — srgan_2016.pdf, page 7

Provider  : mock / gpt-4o-mini
Latency   : 0 ms
Tokens    : 822 (prompt=762, completion=60)
Est. cost : $0.000150
Citations : VALID
  Unused sources: [3, 4, 5]



In [4]:
result = rag_query('How does SwinIR use transformer attention for super-resolution?')

Q: How does SwinIR use transformer attention for super-resolution?
Based on the provided context, the super-resolution methods use various loss functions and architectural components. [1] describes the use of perceptual loss combined with adversarial training. [2] introduces residual channel attention to focus on informative features. This is a mock response — set LLM_PROVIDER in .env to get real answers.

**References**
[1] HAT (, 2022) — hat_2022.pdf, page 1
[2] HAT (, 2022) — hat_2022.pdf, page 13
[3] SwinIR (, 2021) — swinir_2021.pdf, page 1
[4] SwinIR (, 2021) — swinir_2021.pdf, page 1
[5] SwinIR (, 2021) — swinir_2021.pdf, page 5

Provider  : mock / gpt-4o-mini
Latency   : 0 ms
Tokens    : 773 (prompt=713, completion=60)
Est. cost : $0.000143
Citations : VALID
  Unused sources: [3, 4, 5]



In [5]:
result = rag_query('What is the difference between SRGAN and ESRGAN?')

Q: What is the difference between SRGAN and ESRGAN?
Based on the provided context, the super-resolution methods use various loss functions and architectural components. [1] describes the use of perceptual loss combined with adversarial training. [2] introduces residual channel attention to focus on informative features. This is a mock response — set LLM_PROVIDER in .env to get real answers.

**References**
[1] ESRGAN (, 2018) — esrgan_2018.pdf, page 10
[2] ESRGAN (, 2018) — esrgan_2018.pdf, page 11
[3] SRGAN / SRResNet (, 2016) — srgan_2016.pdf, page 8
[4] ESRGAN (, 2018) — esrgan_2018.pdf, page 12
[5] SRGAN / SRResNet (, 2016) — srgan_2016.pdf, page 7

Provider  : mock / gpt-4o-mini
Latency   : 0 ms
Tokens    : 693 (prompt=633, completion=60)
Est. cost : $0.000131
Citations : VALID
  Unused sources: [3, 4, 5]



In [6]:
result = rag_query('Which papers evaluate on the DIV2K dataset?')

Q: Which papers evaluate on the DIV2K dataset?
Based on the provided context, the super-resolution methods use various loss functions and architectural components. [1] describes the use of perceptual loss combined with adversarial training. [2] introduces residual channel attention to focus on informative features. This is a mock response — set LLM_PROVIDER in .env to get real answers.

**References**
[1] EDSR (, 2017) — edsr_2017.pdf, page 7
[2] RDN (, 2018) — rdn_2018.pdf, page 5
[3] EDSR (, 2017) — edsr_2017.pdf, page 4
[4] ESRGAN (, 2018) — esrgan_2018.pdf, page 21
[5] EDSR (, 2017) — edsr_2017.pdf, page 5

Provider  : mock / gpt-4o-mini
Latency   : 0 ms
Tokens    : 819 (prompt=759, completion=60)
Est. cost : $0.000150
Citations : VALID
  Unused sources: [3, 4, 5]



In [7]:
question = 'What is residual channel attention in RCAN?'
qv = embed_query(question)
chunks = query_collection(collection, qv, top_k=3)

prompt, included = build_prompt(question, chunks)
print(f'Prompt length : {len(prompt)} chars')
print(f'Chunks used   : {len(included)}')
print()
print(prompt)

Prompt length : 2875 chars
Chunks used   : 3

Context from research papers:

[1] RCAN (2018) — rcan_2018.pdf, page 8
Fg,b = Fg,b−1 + Rg,b (Xg,b) · Xg,b,
(13)
where Rg,b denotes the function of channel attention. Fg,b and Fg,b−1 are the
input and output of RCAB, which learns the residual Xg,b from the input. The
residual component is mainly obtained by two stacked Conv layers
Xg,b = W 2
g,bδ
 W 1
g,bFg,b−1

,
(14)
where W 1
g,b and W 2
g,b are weight sets the two stacked Conv layers in RCAB.
We further show the relationships between our proposed RCAB and residual
block (RB) in [10]. We find that the RBs used in MDSR and EDSR [10] can be
viewed as special cases of our RCAB. For RB in MDSR, there is no rescaling
operation. It is the same as RCAB, where we set Rg,b (·) as constant 1. For RB
with constant rescaling (e.g., 0.1) in EDSR, it is the same as RCAB with Rg,b (·)
set to be 0.1. Although the channel-wise feature rescaling is introduced to train

---

[2] RCAN (2018) — rcan_2018.pdf

In [8]:
test_questions = [
    'What loss function does SRGAN use?',
    'How does EDSR differ from SRResNet?',
    'What datasets does RCAN evaluate on?',
    'Explain the RRDB architecture in ESRGAN.',
    'What makes RealESRGAN different from ESRGAN?',
]

rows = []
for q in test_questions:
    qv = embed_query(q)
    chunks = query_collection(collection, qv, top_k=5)
    result = generate_answer(q, chunks, provider='mock')
    report = validate_citations(result['answer'], result['sources'])
    rows.append({
        'Question':    q[:50],
        'Chunks':      len(result['sources']),
        'Tokens':      result['token_usage']['total_tokens'],
        'Latency ms':  result['latency_ms'],
        'Cost $':      result['token_usage']['estimated_cost_usd'],
        'Cites valid': report['valid'],
    })

import pandas as pd
pd.DataFrame(rows)

,Question,Chunks,Tokens,Latency ms,Cost $,Cites valid
0,What loss function does SRGAN use?,5,822,0.0,0.000150,True
1,How does EDSR differ from SRResNet?,5,792,0.0,0.000146,True
2,What datasets does RCAN evaluate on?,5,870,0.0,0.000158,True
3,Explain the RRDB architecture in ESRGAN.,5,684,0.0,0.000130,True
4,What makes RealESRGAN different from ESRGAN?,5,578,0.0,0.000114,True


In [9]:
# Uncomment when you have an OpenAI key set in .env:
#
# from dotenv import load_dotenv
# load_dotenv('../.env')
#
# result = rag_query(
#     'What loss function does SRGAN use?',
#     provider='openai',
# )

print('Skipping real LLM call — set LLM_PROVIDER in .env when ready.')

Skipping real LLM call — set LLM_PROVIDER in .env when ready.


In [10]:
import json
from pathlib import Path

log_path = Path('../logs/query_log.jsonl')
if log_path.exists():
    records = [json.loads(line) for line in log_path.read_text().splitlines() if line.strip()]
    print(f'{len(records)} records in query log')
    for r in records[-3:]:
        print(f"  [{r['timestamp']}] {r['question'][:50]} | {r['latency_ms']} ms | {r['token_usage']['total_tokens']} tokens")
else:
    print('No log file yet — run a query first.')

7 records in query log
  [2026-05-22T23:17:36Z] test | 0.0 ms | 168 tokens
  [2026-05-22T23:17:36Z] test | 0.0 ms | 168 tokens
  [2026-05-22T23:17:36Z] test | 0.0 ms | 74 tokens
